
# explore_external_sources_public.ipynb

**Scope:** Open/public data sources **only** (no keys): **USGS**, **OECD/IMF**, **World Bank**.  
Use this to validate endpoints, normalize schemas, and generate CSVs for **Spec 1.1 — External Data Extraction & Ingestion**.

**What you get**
- Fetch scaffolding for USGS (Excel), World Bank (Pink Sheet via API/CSV), OECD & IMF (SDMX JSON/CSV)
- Mock fallbacks if offline
- Normalization to tidy tables (`date`, `indicator`, `value`, `source`)
- Validation checks and quick plots
- Outputs to `data/raw`, `data/processed`, and `config/external_sources_public.yaml`


In [5]:

# 1) Setup
import os, io, json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import datetime as dt

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (9,3)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed'
META_DIR = BASE_DIR / 'config'
for d in (DATA_DIR, RAW_DIR, PROC_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

def fetch_json(url, params=None, headers=None, timeout=45):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print('[WARN] JSON fetch failed:', e)
        return None

def fetch_csv(url, params=None, headers=None, timeout=45):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        import pandas as pd
        return pd.read_csv(io.StringIO(r.text))
    except Exception as e:
        print('[WARN] CSV fetch failed:', e)
        return None

def write_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f'[OK] wrote {len(df):,} rows -> {path}')



## 2) World Bank — Pink Sheet (open)

World Bank Pink Sheet provides monthly commodity prices (including many metals).  
We scaffold a call (replace with exact series you need) and include an offline mock.


In [6]:

# World Bank example (placeholder indicator; replace with actual Pink Sheet metal codes)
WB_URL = 'https://api.worldbank.org/v2/country/WLD/indicator/CM.MKT.INDX.ZG?format=json'  # placeholder
wb_json = fetch_json(WB_URL)

if wb_json and isinstance(wb_json, list) and len(wb_json) > 1 and isinstance(wb_json[1], list):
    rows = wb_json[1]
    df_wb = pd.DataFrame([{
        'date': r.get('date'),
        'value': r.get('value'),
        'indicator': r.get('indicator', {}).get('id', 'WB_UNKNOWN')
    } for r in rows])
    df_wb['date'] = pd.to_datetime(df_wb['date'], errors='coerce')
    df_wb['value'] = pd.to_numeric(df_wb['value'], errors='coerce')
    df_wb.dropna(inplace=True)
    df_wb['source'] = 'WorldBank'
    print('[OK] World Bank live sample fetched.')
else:
    # MOCK: monthly aluminum & copper prices
    dates = pd.date_range('2014-01-01', periods=140, freq='MS')
    df_wb = pd.DataFrame({
        'date': np.tile(dates, 2),
        'indicator': np.repeat(['WB_PINK_ALUMINUM_USD_TON','WB_PINK_COPPER_USD_TON'], len(dates)),
        'value': np.r_[2000 + np.sin(np.linspace(0,16,len(dates)))*70 + np.random.normal(0,10,len(dates)),
                       7000 + np.cos(np.linspace(0,16,len(dates)))*160 + np.random.normal(0,25,len(dates))],
        'source': 'WorldBank_MOCK'
    })
    print('[MOCK] Generated World Bank Pink Sheet metals.')

write_df(df_wb, RAW_DIR / 'worldbank_pink_sheet_raw.csv')

# Normalize to month-end
df_wb_norm = df_wb.copy()
df_wb_norm['date'] = pd.to_datetime(df_wb_norm['date']).dt.to_period('M').dt.to_timestamp('M')
write_df(df_wb_norm, PROC_DIR / 'worldbank_pink_sheet_norm.csv')

# quick look
df_wb_norm.head()


[OK] World Bank live sample fetched.
[OK] wrote 0 rows -> /Users/matthewtonks/Repositories/BUS 659/Class Folder/Final Project/data/raw/worldbank_pink_sheet_raw.csv
[OK] wrote 0 rows -> /Users/matthewtonks/Repositories/BUS 659/Class Folder/Final Project/data/processed/worldbank_pink_sheet_norm.csv


,date,value,indicator,source



## 3) USGS — Metal statistics (open, Excel parsing)

USGS publishes open Excel/CSV/PDF tables. Here we simulate an Excel layout and parsing flow (works offline).
Replace with the real USGS workbook URL when running connected.


In [7]:

import pandas as pd

# Create mock USGS workbook
usgs_xls = RAW_DIR / 'usgs_prices_mock.xlsx'
with pd.ExcelWriter(usgs_xls) as xw:
    sheet = pd.DataFrame({
        'Month': pd.date_range('2016-01-01', periods=120, freq='MS'),
        'Aluminum_USD_Ton': 1800 + np.linspace(0, 250, 120) + np.random.normal(0, 20, 120),
        'Copper_USD_Ton': 6000 + np.linspace(-200, 400, 120) + np.random.normal(0, 35, 120),
        'Nickel_USD_Ton': 16000 + np.linspace(-800, 900, 120) + np.random.normal(0, 120, 120),
    })
    sheet.to_excel(xw, index=False, sheet_name='USGS_Prices')

# Parse mock workbook
df_usgs = pd.read_excel(usgs_xls, sheet_name='USGS_Prices')
df_usgs.rename(columns={'Month':'date'}, inplace=True)
df_usgs = df_usgs.melt(id_vars=['date'], var_name='indicator', value_name='value')
df_usgs['source'] = 'USGS_MOCK'
write_df(df_usgs, RAW_DIR / 'usgs_prices_melted.csv')

# Normalize to month-end
df_usgs_norm = df_usgs.copy()
df_usgs_norm['date'] = pd.to_datetime(df_usgs_norm['date']).dt.to_period('M').dt.to_timestamp('M')
write_df(df_usgs_norm, PROC_DIR / 'usgs_prices_norm.csv')
df_usgs_norm.head()


[OK] wrote 360 rows -> /Users/matthewtonks/Repositories/BUS 659/Class Folder/Final Project/data/raw/usgs_prices_melted.csv
[OK] wrote 360 rows -> /Users/matthewtonks/Repositories/BUS 659/Class Folder/Final Project/data/processed/usgs_prices_norm.csv


,date,indicator,value,source
0,2016-01-31,Aluminum_USD_Ton,1800.905980,USGS_MOCK
1,2016-02-29,Aluminum_USD_Ton,1804.299223,USGS_MOCK
2,2016-03-31,Aluminum_USD_Ton,1804.247501,USGS_MOCK
3,2016-04-30,Aluminum_USD_Ton,1782.004551,USGS_MOCK
4,2016-05-31,Aluminum_USD_Ton,1805.305927,USGS_MOCK



## 4) OECD — SDMX JSON/CSV (open)

OECD stats offer SDMX endpoints. Example scaffold uses the SDMX JSON API.  
Replace `DATASET/FILTERS` with the series you need when online.


In [8]:

# OECD SDMX JSON scaffold (placeholder; offline -> mock)
OECD_URL = 'https://stats.oecd.org/sdmx-json/data/MEI_CLI/OECD.AMPLITUD.M/all?contentType=csv'  # placeholder demo
oecd_csv = fetch_csv(OECD_URL)

if oecd_csv is not None and not oecd_csv.empty:
    df_oecd = oecd_csv.copy()
    # Try to standardize common columns if present
    possible_date_cols = [c for c in df_oecd.columns if c.lower() in ('time','date','period')]
    val_col = next((c for c in df_oecd.columns if c.lower() in ('value','obs_value','obsvalue','data')), None)
    if possible_date_cols and val_col:
        df_oecd = df_oecd.rename(columns={possible_date_cols[0]: 'date', val_col: 'value'})
        df_oecd['date'] = pd.to_datetime(df_oecd['date'], errors='coerce')
        df_oecd['indicator'] = 'OECD_DEMO_SERIES'
        df_oecd['source'] = 'OECD'
        df_oecd = df_oecd[['date','indicator','value','source']].dropna()
        print('[OK] OECD live sample normalized.')
    else:
        # fallback normalization
        df_oecd['indicator'] = 'OECD_RAW'
        df_oecd['source'] = 'OECD'
        print('[OK] OECD live sample (raw).')
else:
    # Mock monthly industrial indicator
    dates = pd.date_range('2014-01-01', periods=140, freq='MS')
    df_oecd = pd.DataFrame({
        'date': dates,
        'indicator': 'OECD_INDICATOR_MOCK',
        'value': 100 + np.sin(np.linspace(0, 20, len(dates))) * 3 + np.random.normal(0, 0.8, len(dates)),
        'source': 'OECD_MOCK'
    })
    print('[MOCK] OECD SDMX demo series.')

write_df(df_oecd, RAW_DIR / 'oecd_series_raw_or_mock.csv')
df_oecd_norm = df_oecd.copy()
df_oecd_norm['date'] = pd.to_datetime(df_oecd_norm['date']).dt.to_period('M').dt.to_timestamp('M')
write_df(df_oecd_norm, PROC_DIR / 'oecd_series_norm.csv')
df_oecd_norm.head()


[OK] OECD live sample (raw).
[OK] wrote 233,578 rows -> /Users/matthewtonks/Repositories/BUS 659/Class Folder/Final Project/data/raw/oecd_series_raw_or_mock.csv


KeyError: 'date'


## 5) IMF — SDMX JSON (open)

IMF offers open SDMX JSON APIs (e.g., IFS). Example scaffold shown; mock used offline.  
When connected, replace `IFS/M.US.PCPI_IX` with the exact code you need.


In [ ]:

IMF_URL = 'https://dataservices.imf.org/REST/SDMX_JSON.svc/CompactData/IFS/M.US.PCPI_IX'  # placeholder
imf_json = fetch_json(IMF_URL)

def imf_compact_to_df(js):
    # Minimalistic extractor for IMF CompactData structure
    try:
        data = js['CompactData']['DataSet']['Series']
        obs = data.get('Obs', [])
        records = []
        for o in obs:
            records.append({'date': o.get('@TIME_PERIOD'), 'value': o.get('@OBS_VALUE')})
        df = pd.DataFrame(records)
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df['indicator'] = data.get('@SERIES_CODE', 'IMF_SERIES')
        df['source'] = 'IMF'
        return df.dropna()
    except Exception as e:
        print('[WARN] IMF parse failed:', e)
        return None

if imf_json:
    df_imf = imf_compact_to_df(imf_json)
else:
    df_imf = None

if df_imf is None or df_imf.empty:
    # Mock monthly macro indicator
    dates = pd.date_range('2013-01-01', periods=156, freq='MS')
    df_imf = pd.DataFrame({
        'date': dates,
        'indicator': 'IMF_IFS_MOCK',
        'value': 100 + np.linspace(-2, 4, len(dates)) + np.random.normal(0, 0.7, len(dates)),
        'source': 'IMF_MOCK'
    })
    print('[MOCK] IMF IFS demo series.')
else:
    print('[OK] IMF live sample normalized.')

write_df(df_imf, RAW_DIR / 'imf_ifs_raw_or_mock.csv')
df_imf_norm = df_imf.copy()
df_imf_norm['date'] = pd.to_datetime(df_imf_norm['date']).dt.to_period('M').dt.to_timestamp('M')
write_df(df_imf_norm, PROC_DIR / 'imf_ifs_norm.csv')
df_imf_norm.head()



## 6) Validation & Quick Plots


In [ ]:

def zflag(s, win=24, thresh=4.0):
    z = (s - s.rolling(win).mean())/s.rolling(win).std()
    return (z.abs() > thresh).astype(int)

def validate_tidy(df, name):
    problems = []
    for col in ['date','indicator','value','source']:
        if col not in df.columns:
            problems.append(f'missing column: {col}')
    if df['date'].isna().any():
        problems.append('null dates')
    if (df['value'] <= 0).sum() > 0 and name.lower().find('price')>=0:
        problems.append('non-positive values in price-like series')
    print(f'[{name}] validation:', problems or 'ok')

for name, d in [('WorldBank', df_wb_norm), ('USGS', df_usgs_norm), ('OECD', df_oecd_norm), ('IMF', df_imf_norm)]:
    validate_tidy(d, name)

# Plot one series per source
for name, d in [('WorldBank', df_wb_norm), ('USGS', df_usgs_norm), ('OECD', df_oecd_norm), ('IMF', df_imf_norm)]:
    ex = d.groupby('indicator', as_index=False).head(1)['indicator'].iloc[0]
    subset = d[d['indicator'] == ex]
    subset.plot(x='date', y='value', title=f'{name} example: {ex}')
    plt.show()



## 7) Metadata YAML for open sources


In [ ]:

import yaml
meta = {
  'world_bank': {
    'examples': ['WB_PINK_ALUMINUM_USD_TON','WB_PINK_COPPER_USD_TON'],
    'notes': 'Replace with actual Pink Sheet series codes or download the monthly CSV bundle.'
  },
  'usgs': {
    'examples': ['Aluminum_USD_Ton','Copper_USD_Ton','Nickel_USD_Ton'],
    'notes': 'Point to official USGS XLS/CSV links when online.'
  },
  'oecd': {
    'examples': ['MEI_CLI demo used'],
    'notes': 'Use SDMX JSON/CSV API and select your target dataset/country/frequency.'
  },
  'imf': {
    'examples': ['IFS series (e.g., M.US.PCPI_IX)'],
    'notes': 'IMF SDMX JSON CompactData endpoint; pick appropriate series codes.'
  },
  'schedule': {'daily': '23:30Z', 'monthly': 'T+3d of provider release'}
}
meta_path = META_DIR / 'external_sources_public.yaml'
with open(meta_path, 'w') as f:
    yaml.safe_dump(meta, f, sort_keys=False)
print('[OK] wrote', meta_path)
print(meta)



## 8) Next Steps
- Replace placeholders with the exact **World Bank Pink Sheet** metal series you need (or the CSV bundle URL).  
- Plug real **USGS** workbook links and confirm column mappings.  
- Select concrete **OECD** dataset filters and **IMF** IFS series codes for your macro needs.  
- Promote the working fetch/normalize code into **Spec 1.1** modules (`extract.py`, `transform.py`, `load.py`).  
- Add data-quality thresholds and Airflow/Prefect operators for production.
